In [1]:
!pip install ultralytics
!pip install torchtoolbox

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 13.2 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from torch.nn import MultiheadAttention
from torchvision import transforms
from PIL import Image
import random
import os
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch.nn.functional as F

# ============================================================================
# CONSTANTS
# ============================================================================
CHARACTERS = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
SOS_TOKEN = 36
EOS_TOKEN = 37
PAD_TOKEN = len(CHARACTERS) + 2  # = 38
NUM_CLASSES = len(CHARACTERS) + 3  # = 39 (0-35 chars, 36 SOS, 37 EOS, 38 PAD)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================
def index_to_char(indices, include_special_tokens=False):
    """Chuyển list index thành chuỗi ký tự, dừng lại tại EOS."""
    result = []
    for i in indices:
        i = i.item() if torch.is_tensor(i) else i
        if i == SOS_TOKEN:
            if include_special_tokens:
                result.append('[SOS]')
        elif i == EOS_TOKEN:
            if include_special_tokens:
                result.append('[EOS]')
            break
        elif i == PAD_TOKEN:
            break
        elif 0 <= i < len(CHARACTERS):
            result.append(CHARACTERS[i])
        else:
            if include_special_tokens:
                result.append(f'[UNK_{i}]')
    return ''.join(result)


def char_to_indices(text):
    """Chuyển chuỗi text thành tensor indices với SOS ở đầu, EOS ở cuối."""
    indices = [SOS_TOKEN]
    for c in text:
        if c in CHARACTERS:
            indices.append(CHARACTERS.index(c))
        else:
            indices.append(0)  # Map unknown char to '0'
    indices.append(EOS_TOKEN)
    return torch.tensor(indices, dtype=torch.long)


# Dùng chung cho cả train và val, tránh lặp code
def extract_pred_and_true(pred_indices, true_indices):
    # Pred: cắt tại EOS hoặc PAD (whichever comes first)
    pred_content = []
    for idx in pred_indices:
        if idx == EOS_TOKEN or idx == PAD_TOKEN:
            break
        pred_content.append(idx)
    
    # True: lọc bỏ EOS và PAD, chỉ giữ ký tự thực
    true_content = [x for x in true_indices if x not in (EOS_TOKEN, PAD_TOKEN)]
    
    return pred_content, true_content


def compute_crr(pred_content, true_content):
    total = max(len(pred_content), len(true_content))
    if total == 0:
        return 0, 0
    
    correct = sum(
        p == t for p, t in zip(pred_content, true_content)
    )
    return correct, total


# ============================================================================
# YOLO BACKBONE
# ============================================================================
class YoloBackbone(nn.Module):
    def __init__(self, model_path):
        super(YoloBackbone, self).__init__()
        self.yolo = YOLO(model_path)
        self.yolo_model = self.yolo.model.to(DEVICE)
        for param in self.yolo_model.parameters():
            param.requires_grad = False  # Không cần huấn luyện YOLO

    def forward(self, x):
        with torch.no_grad():
            results = self.yolo(x, verbose= False)
        boxes = [result.boxes.xyxy.cpu().numpy() for result in results]
        confidences = [result.boxes.conf.cpu().numpy() for result in results]
        return boxes, confidences
        
    def train(self, mode: bool = True):
        self.training = False  # Luôn giữ YOLO ở chế độ eval vì đóng băng
        self.yolo_model.train(False)  # Chỉ gọi train trên yolo_model, không phải yolo
        return self

    def eval(self):
        self.training = False
        self.yolo_model.eval()
        return self 

# ============================================================================
# RViT (Recurrent Vision Transformer)
# ============================================================================
class RViT(nn.Module):
    def __init__(self, yolo_channels=256, d_model=512, num_patches=1600,
                 n_heads=8, num_encoder_layers=3, dim_feedforward=2048,
                 dropout_rate=0.3, max_seq_length=15):
        super().__init__()
        self.d_model = d_model
        self.max_seq_length = max_seq_length
        self.export_mode = False
        
        self.proj = nn.Sequential(
            nn.Conv2d(yolo_channels, d_model, kernel_size=3, padding=1),
            nn.BatchNorm2d(d_model),
            nn.ReLU(),
            nn.Dropout2d(dropout_rate)
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward,
            dropout=dropout_rate, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)

        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, d_model))
        self.region_q = nn.Parameter(torch.zeros(1, 1, d_model))

        self.embed = nn.Embedding(NUM_CLASSES, d_model)
        self.gru_num_layers = 2
        self.gru = nn.GRU(
            d_model, d_model, num_layers=self.gru_num_layers,
            batch_first=True,
            dropout=dropout_rate if self.gru_num_layers > 1 else 0
        )
        self.attn = MultiheadAttention(
            d_model, num_heads=n_heads, batch_first=True, dropout=dropout_rate
        )
        self.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(2 * d_model, NUM_CLASSES)
        )

        self.register_buffer("_sos", torch.tensor(SOS_TOKEN, dtype=torch.long))
        self.register_buffer("_eos", torch.tensor(EOS_TOKEN, dtype=torch.long))

    def _encode(self, fmap):
        b = fmap.size(0)
        x = self.proj(fmap)
        x = x.flatten(2).permute(0, 2, 1)

        n = x.size(1) + 1
        pos = self.pos_embed[:, :n, :]

        q = self.region_q.expand(b, -1, -1)
        x = torch.cat([q, x], dim=1) + pos

        enc = self.encoder(x)
        return enc[:, 0], enc[:, 1:]

    def _decode_step(self, current_token, h, spatial_feats):
        emb = self.embed(current_token).unsqueeze(1)
        g, h = self.gru(emb, h)
        a, _ = self.attn(g, spatial_feats, spatial_feats)
        comb = torch.cat([g.squeeze(1), a.squeeze(1)], dim=-1)
        logits = self.fc(comb)
        return logits, h

    def _decode_train(self, region_feat, spatial_feats, target, teach_ratio):
        b = region_feat.size(0)
        device = region_feat.device
        max_gen_len = target.size(1) - 1

        h = region_feat.unsqueeze(0).repeat(self.gru_num_layers, 1, 1).contiguous()
        current_token = torch.full((b,), SOS_TOKEN, device=device, dtype=torch.long)
        outputs = []

        for t in range(max_gen_len):
            logits, h = self._decode_step(current_token, h, spatial_feats)
            outputs.append(logits)

            if random.random() < teach_ratio:
                current_token = target[:, t + 1]
            else:
                current_token = logits.argmax(-1)

        return torch.stack(outputs, dim=1)

    def _decode_inference(self, region_feat, spatial_feats, forced_output_length=None):
        b = region_feat.size(0)
        device = region_feat.device
        max_gen_len = forced_output_length if forced_output_length else (self.max_seq_length - 1)

        h = region_feat.unsqueeze(0).repeat(self.gru_num_layers, 1, 1).contiguous()
        current_token = torch.full((b,), SOS_TOKEN, device=device, dtype=torch.long)
        outputs = []
        finished = torch.zeros(b, dtype=torch.bool, device=device)

        for t in range(max_gen_len):
            logits, h = self._decode_step(current_token, h, spatial_feats)
            outputs.append(logits)

            next_token = logits.argmax(-1)
            finished |= (next_token == EOS_TOKEN)
            current_token = torch.where(
                finished,
                torch.tensor(EOS_TOKEN, device=device, dtype=torch.long),
                next_token
            )
            if finished.all():
                break

        return torch.stack(outputs, dim=1)

    def forward(self, fmap, target=None, teach_ratio=0.5,
                forced_output_length=None):

        region_feat, spatial_feats = self._encode(fmap)

        if self.export_mode:
            return self._decode_export(region_feat, spatial_feats)

        if target is not None:
            return self._decode_train(region_feat, spatial_feats, target, teach_ratio)

        return self._decode_inference(region_feat, spatial_feats, forced_output_length)

    def _decode_export(self, region_feat, spatial_feats):
        """
        ONNX-friendly decode:
        - Loop cố định, không break
        - Không dùng Python bool branching
        - Kết quả GIỐNG HỆT _decode_inference (greedy argmax)
        """
        b = region_feat.size(0)
        device = region_feat.device
        max_steps = self.max_seq_length - 2

        h = region_feat.unsqueeze(0).expand(
            self.gru_num_layers, -1, -1
        ).contiguous()

        current_token = self._sos.expand(b)
        finished = torch.zeros(b, device=device, dtype=torch.float32)
        all_logits = []

        for t in range(max_steps):
            logits, h = self._decode_step(current_token, h, spatial_feats)
            all_logits.append(logits)

            next_token = logits.argmax(dim=-1)
            is_eos = (next_token == self._eos).float()
            finished = torch.clamp(finished + is_eos, max=1.0)

            current_token = torch.where(
                finished > 0.5,
                self._eos.expand(b),
                next_token
            )

        return torch.stack(all_logits, dim=1)
         
    def prepare_export(self):
        self.export_mode = True
        self.eval()
        return self

    def finish_export(self):
        self.export_mode = False
        return self

# ============================================================================
# FULL MODEL (YOLO_RViT)
# ============================================================================
class YOLO_RViT(nn.Module):
    def __init__(self, yolo_path):
        super().__init__()
        self.detector = YoloBackbone(yolo_path)
        # RViT sẽ nhận đầu vào 3 kênh (ảnh crop RGB đã resize)
        self.rvit = RViT(yolo_channels=3, num_patches=1600).to(DEVICE) # yolo_channels=3 vì input là ảnh crop RGB

    def forward(self, x, target=None, teach_ratio=0.5, forced_output_length=None):
        x = x.to(DEVICE) # x là (B, 3, H_orig, W_orig), ví dụ (B, 3, 640, 640)
        boxes, confidences = self.detector(x)
        cropped_images_list = []

        for i in range(x.size(0)):
            img_h, img_w = x.shape[2], x.shape[3]
        
            if len(boxes[i]) > 0:
                best_idx = np.argmax(confidences[i])
                x1, y1, x2, y2 = map(int, boxes[i][best_idx])
        
                # clamp bbox
                x1 = max(0, min(x1, img_w - 1))
                y1 = max(0, min(y1, img_h - 1))
                x2 = max(x1 + 1, min(x2, img_w))
                y2 = max(y1 + 1, min(y2, img_h))
        
                cropped = x[i][:, y1:y2, x1:x2]
            else:
                cropped = x[i]
        
            cropped_resized = resize_with_padding(cropped, (40, 40))
            cropped_resized = (cropped_resized - torch.tensor([0.485,0.456,0.406], device=cropped.device).view(3,1,1)) / torch.tensor([0.229,0.224,0.225], device=cropped.device).view(3,1,1)
            cropped_images_list.append(cropped_resized)

        cropped_images_tensor = torch.stack(cropped_images_list).to(DEVICE) # Sẽ có shape (B, 3, 40, 40)
        cropped_images_tensor = cropped_images_tensor.to(next(self.rvit.parameters()).dtype)

        return self.rvit(cropped_images_tensor, target, teach_ratio, forced_output_length)
        

def resize_with_padding(image_tensor, target_size=(160, 160)):
    C, H, W = image_tensor.shape
    target_h, target_w = target_size

    # --- scale giữ aspect ratio ---
    scale = min(target_w / W, target_h / H)
    new_w = int(W * scale)
    new_h = int(H * scale)

    # --- resize ---
    image_resized = F.interpolate(
        image_tensor.unsqueeze(0),
        size=(new_h, new_w),
        mode='bilinear',
        align_corners=False
    ).squeeze(0)

    # --- padding ---
    pad_w = target_w - new_w
    pad_h = target_h - new_h

    # chia đều padding (center)
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left
    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top

    image_padded = F.pad(
        image_resized,
        (pad_left, pad_right, pad_top, pad_bottom),  # (left, right, top, bottom)
        mode='constant',
        value=0  # nền đen
    )

    return image_padded

# ============================================================================
# DATASET
# ============================================================================
class LicensePlateDataset(Dataset):
    def __init__(self, img_dir, license_dir, max_seq_length=15, is_train=True):
        self.img_dir = img_dir
        self.license_dir = license_dir
        self.max_seq_length = max_seq_length
        self.img_names = [f for f in os.listdir(self.img_dir) if f.endswith('.jpg')]

        if is_train:
            self.transform = transforms.Compose([
                transforms.Resize((640, 640)),
                transforms.RandomApply([
                    transforms.RandomRotation(10),
                    transforms.RandomAffine(degrees=8, translate=(0.03, 0.03), scale=(0.95, 1.05)),
                    transforms.RandomPerspective(distortion_scale=0.05, p=0.3),
                ], p=0.7),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
                transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.25),
                transforms.ToTensor(),
                transforms.RandomErasing(p=0.2, scale=(0.01, 0.04), ratio=(0.3, 3.3)),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((640, 640)),
                transforms.ToTensor(),
            ])

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        img = Image.open(img_path).convert("RGB")
        img_tensor = self.transform(img)

        license_filename = os.path.splitext(self.img_names[idx])[0] + ".txt"
        license_path = os.path.join(self.license_dir, license_filename)

        with open(license_path, 'r', encoding='utf-8') as f:
            license_text = f.read().upper().strip()

        license_indices = char_to_indices(license_text)
        target = torch.full((self.max_seq_length,), PAD_TOKEN, dtype=torch.long)

        actual_len = min(len(license_indices), self.max_seq_length)
        target[:actual_len] = license_indices[:actual_len]

        return img_tensor, target

    @staticmethod
    def collate_fn(batch):
        images = torch.stack([item[0] for item in batch])
        targets = torch.stack([item[1] for item in batch])
        return images, targets

# ============================================================================
# HYPERPARAMETERS
# ============================================================================
YOLO_MODEL_PATH = '/kaggle/input/models/thiettnnnnnn/t-yolov11s-vnlp/pytorch/default/1/last.pt'

IMG_DIR_TRAIN = "/kaggle/input/datasets/jtachitv/datasetvn2/DatasetVN2/images/train"
LICENSE_DIR_TRAIN = "/kaggle/input/datasets/jtachitv/datasetvn2/DatasetVN2/texts/train"

IMG_DIR_VAL = "/kaggle/input/datasets/jtachitv/datasetvn2/DatasetVN2/images/valid"
LICENSE_DIR_VAL = "/kaggle/input/datasets/jtachitv/datasetvn2/DatasetVN2/texts/valid"

MAX_SEQ_LENGTH = 15
BATCH_SIZE = 32
NUM_WORKERS = 4
LEARNING_RATE = 5e-5
MAX_LR_SCHEDULER = 5e-4
WEIGHT_DECAY = 5e-5
NUM_EPOCHS = 200
ACCUM_STEPS = 2
PATIENCE_EARLY_STOP = 7
TEACH_RATIO_START = 0.7
TEACH_RATIO_END = 0.05
LABEL_SMOOTHING = 0.1

# ============================================================================
# SETUP
# ============================================================================
scaler = torch.amp.GradScaler(DEVICE)
autocast_context = lambda: torch.amp.autocast(DEVICE)

train_dataset_full = LicensePlateDataset(
    img_dir=IMG_DIR_TRAIN, license_dir=LICENSE_DIR_TRAIN,
    max_seq_length=MAX_SEQ_LENGTH, is_train=True
)
val_dataset = LicensePlateDataset(
    img_dir=IMG_DIR_VAL, license_dir=LICENSE_DIR_VAL,
    max_seq_length=MAX_SEQ_LENGTH, is_train=False
)

train_dataset = Subset(train_dataset_full, list(range(50)))

train_dataloader = DataLoader(
    train_dataset_full, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=LicensePlateDataset.collate_fn,
    pin_memory=(DEVICE == 'cuda'), drop_last=True
)
val_dataloader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=LicensePlateDataset.collate_fn,
    pin_memory=(DEVICE == 'cuda')
)

model = YOLO_RViT(
    YOLO_MODEL_PATH,
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Mỗi accumulation cycle = 1 optimizer step
# Số batch cuối cùng cũng trigger 1 step nếu không chia hết
num_batches = len(train_dataloader)
steps_per_epoch = (num_batches + ACCUM_STEPS - 1) // ACCUM_STEPS

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=MAX_LR_SCHEDULER,
    epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    pct_start=0.2,
    div_factor=(MAX_LR_SCHEDULER / LEARNING_RATE) if MAX_LR_SCHEDULER > LEARNING_RATE else 10.0
)
scheduler_type = "OneCycleLR"

loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN, label_smoothing=LABEL_SMOOTHING)

# ============================================================================
# TRAINING LOOP
# ============================================================================
train_loss_values, val_loss_values = [], []
train_acc_values, val_acc_values, val_acc_sequence_values = [], [], []
epoch_count_list = []
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    epoch_count_list.append(epoch + 1)

    # --- Teach ratio decay ---
    teach_ratio = TEACH_RATIO_START - (TEACH_RATIO_START - TEACH_RATIO_END) * (epoch / max(1, NUM_EPOCHS - 1))

    # ================================================================
    # TRAIN PHASE
    # ================================================================
    model.train()
    train_loss, train_correct, train_total_chars = 0, 0, 0

    pbar_train = tqdm(
        train_dataloader,
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [TRAIN] LR: {optimizer.param_groups[0]['lr']:.2e} "
             f"Teach: {teach_ratio:.2f} Scheduler: {scheduler_type}"
    )

    for batch_idx, (imgs, targets) in enumerate(pbar_train):
        imgs = imgs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        # --- Forward + Loss ---
        with autocast_context():
            outputs = model(imgs, target=targets, teach_ratio=teach_ratio)
            # outputs: [B, seq_len, NUM_CLASSES], targets[:, 1:]: [B, seq_len]
            flat_outputs = outputs.reshape(-1, NUM_CLASSES)
            flat_targets = targets[:, 1:].reshape(-1)
            loss = loss_fn(flat_outputs, flat_targets)
            loss = loss / ACCUM_STEPS

        # --- Backward ---
        scaler.scale(loss).backward()

        # --- Optimizer step (gradient accumulation) ---
        if (batch_idx + 1) % ACCUM_STEPS == 0 or (batch_idx + 1) == num_batches:
            torch.nn.utils.clip_grad_norm_(
                (p for p in model.parameters() if p.requires_grad),
                max_norm=1.0
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if scheduler_type == "OneCycleLR":
                scheduler.step()

        # --- Train metrics (không cần gradient) ---
        train_loss += loss.item() * ACCUM_STEPS
        with torch.no_grad():
            preds = outputs.argmax(-1)
            true_chars = targets[:, 1:]

            for i in range(imgs.size(0)):
                pred_content, true_content = extract_pred_and_true(
                    preds[i].tolist(), true_chars[i].tolist()
                )
                correct, total = compute_crr(pred_content, true_content)
                train_correct += correct
                train_total_chars += total

            # --- Print examples (batch 0 mỗi epoch) ---
            if batch_idx == 0 and imgs.size(0) > 0:
                print("\n--- Training Batch 0 Examples ---")
                for i in range(min(5, imgs.size(0))):
                    pred_example = index_to_char(preds[i].tolist())
                    true_example = index_to_char(true_chars[i].tolist())
                    print(f"  Pred: '{pred_example}'")
                    print(f"  True: '{true_example}'")
                print("-------------------------------")

        pbar_train.set_postfix(loss=loss.item() * ACCUM_STEPS)

    avg_train_loss = train_loss / num_batches if num_batches > 0 else 0
    avg_train_acc = train_correct / train_total_chars if train_total_chars > 0 else 0
    train_loss_values.append(avg_train_loss)
    train_acc_values.append(avg_train_acc)

    # ================================================================
    # VALIDATION PHASE
    # ================================================================
    model.eval()
    val_loss = 0
    val_correct, val_total_chars = 0, 0
    val_correct_sequences, val_total_sequences = 0, 0
    num_samples = 0

    pbar_val = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [VAL]")

    with torch.no_grad():
        for imgs, targets in pbar_val:
            imgs = imgs.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            batch_size = imgs.size(0)
            num_samples += batch_size

            with autocast_context():
                outputs = model(imgs, target=None, teach_ratio=0.0)
            
            with autocast_context():
                out_seq_len = outputs.size(1)
                tgt_content_len = targets.size(1) - 1  # Bỏ SOS ở đầu target

                # Lấy phần ngắn hơn giữa output và target
                min_len = min(out_seq_len, tgt_content_len)
                outputs_for_loss = outputs[:, :min_len, :]
                targets_for_loss = targets[:, 1:min_len + 1]  # Bỏ SOS, lấy min_len tokens

                flat_outputs_val = outputs_for_loss.reshape(-1, NUM_CLASSES)
                flat_targets_val = targets_for_loss.reshape(-1)
                loss = loss_fn(flat_outputs_val, flat_targets_val)

            val_loss += loss.item()

            preds_val = outputs.argmax(-1) 
            true_chars_val = targets[:, 1:]

            for i in range(batch_size):
                pred_content, true_content = extract_pred_and_true(
                    preds_val[i].tolist(), true_chars_val[i].tolist()
                )

                # CRR
                correct, total = compute_crr(pred_content, true_content)
                val_correct += correct
                val_total_chars += total

                # E2E exact match (toàn bộ biển số phải đúng hoàn toàn)
                if pred_content == true_content:
                    val_correct_sequences += 1
                val_total_sequences += 1

            pbar_val.set_postfix(loss=loss.item())

    # ================================================================
    # EPOCH SUMMARY
    # ================================================================
    avg_val_loss = val_loss / len(val_dataloader) if len(val_dataloader) > 0 else 0
    avg_val_acc = val_correct / val_total_chars if val_total_chars > 0 else 0
    avg_val_sequence_accuracy = val_correct_sequences / val_total_sequences if val_total_sequences > 0 else 0.0

    val_loss_values.append(avg_val_loss)
    val_acc_values.append(avg_val_acc)
    val_acc_sequence_values.append(avg_val_sequence_accuracy)

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} | LR: {optimizer.param_groups[0]['lr']:.2e} "
          f"| Teach: {teach_ratio:.2f} | Scheduler: {scheduler_type}")
    print(f"  Train Loss: {avg_train_loss:.4f} | Train CRR: {avg_train_acc:.4f}")
    print(f"  Val Loss:   {avg_val_loss:.4f} | Val CRR:   {avg_val_acc:.4f}")
    print(f"  Val E2E RR: {avg_val_sequence_accuracy:.4f}")
    print("-" * 70)

    # --- Save best model ---
    if avg_val_acc > best_val_acc:
        best_val_acc = avg_val_acc
        print(f"*** New best CRR: {best_val_acc:.4f}. Saving best_model.pth ***")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_loss': avg_val_loss,
            'val_crr': avg_val_acc,
            'val_e2e_rr': avg_val_sequence_accuracy,
        }, "best_yolo_rvit_model.pth")

# ============================================================================
# SAVE FINAL MODEL
# ============================================================================
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'train_loss_history': train_loss_values,
    'val_loss_history': val_loss_values,
    'train_acc_history': train_acc_values,
    'val_acc_history': val_acc_values,
    'val_acc_sequence_history': val_acc_sequence_values,
}, "final_yolo_rvit_model.pth")

print("\nTraining completed!")
if val_acc_values:
    print(f"Final Val CRR:    {val_acc_values[-1]:.4f}")
if val_acc_sequence_values:
    print(f"Final Val E2E RR: {val_acc_sequence_values[-1]:.4f}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/tmp/ipykernel_23/2732481682.py:138: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
Epoch 1/200 [TRAIN] LR: 5.00e-05 Teach: 0.70 Scheduler: OneCycleLR:   1%|          | 1/95 [00:16<25:25, 16.23s/it, loss=3.69]


--- Training Batch 0 Examples ---
  Pred: '0EEJJKEESNEPBE'
  True: '72C05351'
  Pred: 'NQQAAQQQQQ6QJA'
  True: '66P15739'
  Pred: 'QKE66Q1Q66QQU6'
  True: '59C162293'
  Pred: 'EMAJ6G8JEEKEJE'
  True: '49C09971'
  Pred: 'AKNECSSYPCEI8'
  True: '54U39055'
-------------------------------


Epoch 1/200 [TRAIN] LR: 5.00e-05 Teach: 0.70 Scheduler: OneCycleLR: 100%|██████████| 95/95 [01:59<00:00,  1.26s/it, loss=2.82]
Epoch 1/200 [VAL]: 100%|██████████| 56/56 [00:31<00:00,  1.79it/s, loss=2.65]



Epoch 1/200 | LR: 5.07e-05 | Teach: 0.70 | Scheduler: OneCycleLR
  Train Loss: 3.0853 | Train CRR: 0.1154
  Val Loss:   2.8000 | Val CRR:   0.0984
  Val E2E RR: 0.0000
----------------------------------------------------------------------
*** New best CRR: 0.0984. Saving best_model.pth ***


...


Epoch 199/200 | LR: 5.12e-08 | Teach: 0.05 | Scheduler: OneCycleLR
  Train Loss: 1.2398 | Train CRR: 0.7903
  Val Loss:   1.1490 | Val CRR:   0.8340
  Val E2E RR: 0.4748
----------------------------------------------------------------------


Epoch 200/200 [TRAIN] LR: 5.12e-08 Teach: 0.05 Scheduler: OneCycleLR:   1%|          | 1/95 [00:05<08:36,  5.49s/it, loss=1.22]


--- Training Batch 0 Examples ---
  Pred: '53F34466'
  True: '93F94406'
  Pred: '59L217560'
  True: '59L227560'
  Pred: '54L38967'
  True: '54L36961'
  Pred: '51C72891'
  True: '51C72891'
  Pred: '49R00165'
  True: '49R00165'
-------------------------------


Epoch 200/200 [TRAIN] LR: 5.12e-08 Teach: 0.05 Scheduler: OneCycleLR: 100%|██████████| 95/95 [01:55<00:00,  1.21s/it, loss=1.26]
Epoch 200/200 [VAL]: 100%|██████████| 56/56 [00:30<00:00,  1.86it/s, loss=1.11]



Epoch 200/200 | LR: 5.02e-09 | Teach: 0.05 | Scheduler: OneCycleLR
  Train Loss: 1.2336 | Train CRR: 0.7945
  Val Loss:   1.1504 | Val CRR:   0.8347
  Val E2E RR: 0.4776
----------------------------------------------------------------------

Training completed!
Final Val CRR:    0.8347
Final Val E2E RR: 0.4776


In [3]:
!pip install onnxscript
model.eval()
model.rvit.prepare_export()

dummy_crop = torch.randn(1, 3, 40, 40, device=DEVICE)

torch.onnx.export(
    model.rvit,
    (dummy_crop,),
    "/kaggle/working/rvit_2stage.onnx",
    input_names=["crop"],
    output_names=["ocr_logits"],
    dynamic_axes={
        "crop":       {0: "batch"},
        "ocr_logits": {0: "batch"},
    },
    opset_version=20,          # ← đổi 17 → 18
    do_constant_folding=True,
    dynamo=False,              # ← tắt dynamo, dùng TorchScript trace cũ
)

model.rvit.finish_export()
print("Done — rvit_2stage.onnx")

import shutil
shutil.copy(YOLO_MODEL_PATH, "/kaggle/working/yolo2stage.pt")

yolo = YOLO("/kaggle/working/yolo2stage.pt")
yolo.export(format="onnx", imgsz=640)
# → save ra /kaggle/working/yolo2stage.onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 11.2 MB/s eta 0:00:00


/tmp/ipykernel_23/2818643966.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:4553: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with GRU can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


Done — rvit_2stage.onnx
Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs

PyTorch: starting from '/kaggle/working/yolo2stage.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (18.3 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 266ms
 Downloaded onnxruntime-gpu
Prepared 2 packages in 2.51s
Installed 2 packages in 15ms
 + onnxruntime-gpu==1.24.4
 + onnxslim==0.1.91

requirements: AutoUpdate success ✅ 3.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 20...
ONNX: slimming with onnxsli

'/kaggle/working/yolo2stage.onnx'

In [4]:
!pip install tensorrt --break-system-packages
import tensorrt as trt

def onnx_to_trt(onnx_path, engine_path, fp16=True):
    logger = trt.Logger(trt.Logger.INFO)
    builder = trt.Builder(logger)
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    parser = trt.OnnxParser(network, logger)

    with open(onnx_path, 'rb') as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(parser.get_error(i))
            return

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)

    if fp16 and builder.platform_has_fast_fp16:
        config.set_flag(trt.BuilderFlag.FP16)

    profile = builder.create_optimization_profile()
    for i in range(network.num_inputs):
        inp = network.get_input(i)
        shape = list(inp.shape)
        fixed_shape = [1 if d == -1 else d for d in shape]

        profile.set_shape(
            inp.name,
            fixed_shape,
            fixed_shape,
            fixed_shape
        )
    config.add_optimization_profile(profile)

    print(f"Building {engine_path}...")
    engine = builder.build_serialized_network(network, config)
    with open(engine_path, 'wb') as f:
        f.write(engine)
    print(f"Done! ({engine.nbytes/1024/1024:.1f} MB)")

onnx_to_trt("/kaggle/working/rvit_2stage.onnx", "/kaggle/working/rvit_2stage.engine")
onnx_to_trt("/kaggle/working/yolo2stage.onnx", "/kaggle/working/yolo_2stage.engine")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.9 MB/s eta 0:00:00
  Created wheel for tensorrt: filename=tensorrt-10.16.1.11-py3-none-any.whl size=16564 sha256=ebb3475541e5445984e57302f5ee39b250dbca17ff0b52bdda50cdc6aacee400
  Stored in directory: /root/.cache/pip/wheels/9d/0c/7c/76b5862f9a4b940416c6277c77feb266b16b842f1cb26d8f9b
  Created wheel for tensorrt_cu13: filename=tensorrt_cu13-10.16.1.11-py3-none-any.whl size=23075 sha256=94ce8bcc3086d9e8aa37b4f35508b90511bbbb49e0ac5e78635d103e51d08046
  Stored in directory: /root/.cache/pip/wheels/4f/55/9a/84d81786d3b4213025298a1ed9b

In [5]:
# ============================================================================
# MODEL COMPLEXITY & BENCHMARK (batch_size=1)
# ============================================================================
import numpy as np
from torch.utils.flop_counter import FlopCounterMode

model.eval()

# ============================
# 1. PARAMETER COUNT
# ============================
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
backbone_params = sum(p.numel() for p in model.detector.parameters())
rvit_params = sum(p.numel() for p in model.rvit.parameters())

# ============================
# 2. MODEL SIZE (FP32 / FP16)
# ============================
model_size_fp32_mb = total_params * 4 / (1024 ** 2)
model_size_fp16_mb = total_params * 2 / (1024 ** 2)

print("=" * 70)
print("MODEL COMPLEXITY")
print("=" * 70)
print(f"  Total params:      {total_params / 1e6:.2f} M")
print(f"  Trainable params:  {trainable_params / 1e6:.2f} M")
print(f"  Backbone (YOLO):   {backbone_params / 1e6:.2f} M")
print(f"  RVT (ViT+GRU):     {rvit_params / 1e6:.2f} M")
print(f"  Model size FP32:   {model_size_fp32_mb:.2f} MB")
print(f"  Model size FP16:   {model_size_fp16_mb:.2f} MB")

# ============================
# 3. FLOPs
# ============================
dummy_input = torch.randn(1, 3, 640, 640).to(DEVICE)

with torch.inference_mode():
    flop_counter = FlopCounterMode(display=False)
    with flop_counter:
        _ = model(dummy_input, target=None, teach_ratio=0.0,
                  forced_output_length=MAX_SEQ_LENGTH - 1)
    total_flops = flop_counter.get_total_flops()

print(f"  FLOPs @640x640:    {total_flops / 1e9:.2f} GFLOPs")
print("=" * 70)

# ============================================================================
# BENCHMARK FP32 (batch_size=1)
# ============================================================================
test_dataloader_batch1 = DataLoader(
    val_dataset, batch_size=1, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=LicensePlateDataset.collate_fn,
    pin_memory=(DEVICE == 'cuda')
)

latencies_fp32 = []
NUM_WARMUP = 50

with torch.inference_mode():
    for i, (imgs, targets) in enumerate(tqdm(test_dataloader_batch1, desc="[BENCHMARK FP32]")):
        imgs = imgs.to(DEVICE, non_blocking=True)

        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)

        start_event.record()
        outputs = model(imgs, target=None, teach_ratio=0.0)
        end_event.record()
        torch.cuda.synchronize()

        elapsed_ms = start_event.elapsed_time(end_event)
        if i > NUM_WARMUP:
            latencies_fp32.append(elapsed_ms)

latencies_fp32 = np.array(latencies_fp32)

mean_latency_fp32 = np.mean(latencies_fp32)
std_latency_fp32 = np.std(latencies_fp32)
median_latency_fp32 = np.median(latencies_fp32)
fps_fp32 = 1000.0 / mean_latency_fp32

print("\n" + "=" * 70)
print("📊 BENCHMARK RESULTS (FP32, batch_size=1, torch.compile, T4 GPU)")
print("=" * 70)
print(f"  Num samples:      {len(latencies_fp32)} (sau {NUM_WARMUP} warm-up)")
print(f"  Mean latency:     {mean_latency_fp32:.2f} ± {std_latency_fp32:.2f} ms")
print(f"  Median latency:   {median_latency_fp32:.2f} ms")
print(f"  FPS:              {fps_fp32:.1f}")
print("=" * 70)

# ============================
# BENCHMARK FP16 — không dùng torch.compile
# ============================
# model.eval()

# latencies_fp16 = []
# NUM_WARMUP = 50

# with torch.inference_mode():
#     for i, (imgs, targets) in enumerate(tqdm(test_dataloader_batch1, desc="[BENCHMARK FP16]")):
#         imgs = imgs.to(DEVICE, non_blocking=True)

#         start_event = torch.cuda.Event(enable_timing=True)
#         end_event = torch.cuda.Event(enable_timing=True)

#         start_event.record()
#         with torch.amp.autocast('cuda', dtype=torch.float16):  # ← FP16 tại đây
#             outputs = model(imgs, target=None, teach_ratio=0.0)
#         end_event.record()
#         torch.cuda.synchronize()

#         elapsed_ms = start_event.elapsed_time(end_event)
#         if i >= NUM_WARMUP:
#             latencies_fp16.append(elapsed_ms)

# latencies_fp16 = np.array(latencies_fp16)
# mean_latency_fp16 = np.mean(latencies_fp16)
# std_latency_fp16 = np.std(latencies_fp16)
# median_latency_fp16 = np.median(latencies_fp16)
# fps_fp16 = 1000.0 / mean_latency_fp16

# print("\n" + "=" * 70)
# print("BENCHMARK RESULTS (FP16, batch_size=1, autocast, T4 GPU)")
# print("=" * 70)
# print(f"  Samples measured:  {len(latencies_fp16)} (after {NUM_WARMUP} warm-up)")
# print(f"  Mean latency:      {mean_latency_fp16:.2f} ± {std_latency_fp16:.2f} ms")
# print(f"  Median latency:    {median_latency_fp16:.2f} ms")
# print(f"  FPS (bs=1):        {fps_fp16:.1f}")
# print("=" * 70)

MODEL COMPLEXITY
  Total params:      23.97 M
  Trainable params:  14.56 M
  Backbone (YOLO):   9.41 M
  RVT (ViT+GRU):     14.56 M
  Model size FP32:   91.43 MB
  Model size FP16:   45.72 MB
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.930957794189453. Dividing input by 255.
  FLOPs @640x640:    36.66 GFLOPs


[BENCHMARK FP32]: 100%|██████████| 1763/1763 [01:13<00:00, 23.91it/s]


📊 BENCHMARK RESULTS (FP32, batch_size=1, torch.compile, T4 GPU)
  Num samples:      1712 (sau 50 warm-up)
  Mean latency:     40.33 ± 2.72 ms
  Median latency:   39.69 ms
  FPS:              24.8


In [6]:
# ============================================================
# TensorRT 2-Stage: Load 2 engines
# ============================================================
import tensorrt as trt
import numpy as np
import pycuda.driver as cuda
import pycuda.autoinit
import torch
import torch.nn.functional as F

# ── Engine 1: YOLO ──
runtime = trt.Runtime(trt.Logger(trt.Logger.WARNING))
with open("/kaggle/working/yolo_2stage.engine", "rb") as f:
    yolo_engine = runtime.deserialize_cuda_engine(f.read())
yolo_context = yolo_engine.create_execution_context()

# ── Engine 2: RViT ──
with open("/kaggle/working/rvit_2stage.engine", "rb") as f:
    rvit_engine = runtime.deserialize_cuda_engine(f.read())
rvit_context = rvit_engine.create_execution_context()

# ── YOLO setup ──
YOLO_INPUT  = yolo_engine.get_tensor_name(0)
YOLO_OUTPUT = yolo_engine.get_tensor_name(1)
yolo_context.set_input_shape(YOLO_INPUT, (1, 3, 640, 640))
yolo_out_shape = tuple(yolo_context.get_tensor_shape(YOLO_OUTPUT))

d_yolo_in  = cuda.mem_alloc(1 * 3 * 640 * 640 * 4)
d_yolo_out = cuda.mem_alloc(int(np.prod(yolo_out_shape)) * 4)
h_yolo_out = cuda.pagelocked_zeros(yolo_out_shape, dtype=np.float32)  # ★ pinned memory

yolo_context.set_tensor_address(YOLO_INPUT,  int(d_yolo_in))
yolo_context.set_tensor_address(YOLO_OUTPUT, int(d_yolo_out))

# ── RViT setup ──
RVIT_INPUT  = rvit_engine.get_tensor_name(0)
RVIT_OUTPUT = rvit_engine.get_tensor_name(1)
rvit_context.set_input_shape(RVIT_INPUT, (1, 3, 40, 40))
rvit_out_shape = tuple(rvit_context.get_tensor_shape(RVIT_OUTPUT))

d_rvit_in  = cuda.mem_alloc(1 * 3 * 40 * 40 * 4)
d_rvit_out = cuda.mem_alloc(int(np.prod(rvit_out_shape)) * 4)
h_rvit_out = cuda.pagelocked_zeros(rvit_out_shape, dtype=np.float32)  # ★ pinned memory

rvit_context.set_tensor_address(RVIT_INPUT,  int(d_rvit_in))
rvit_context.set_tensor_address(RVIT_OUTPUT, int(d_rvit_out))

# ── Pre-allocate GPU tensors cho crop/resize ──
d_mean = torch.tensor([0.485, 0.456, 0.406], device='cuda').view(1, 3, 1, 1)
d_std  = torch.tensor([0.229, 0.224, 0.225], device='cuda').view(1, 3, 1, 1)

# ★ Pre-allocate pinned memory cho YOLO input (tránh allocate mỗi lần)
h_yolo_in = cuda.pagelocked_zeros((1, 3, 640, 640), dtype=np.float32)

stream = cuda.Stream()

print(f"YOLO: {YOLO_INPUT} → {YOLO_OUTPUT} {yolo_out_shape}")
print(f"RViT: {RVIT_INPUT} → {RVIT_OUTPUT} {rvit_out_shape}")


# ============================================================
# Postprocess YOLO → best box (giữ nguyên)
# ============================================================
def postprocess_yolo(output, conf_thresh=0.25):
    pred = output.squeeze(0)
    if pred.shape[0] < pred.shape[1]:
        pred = pred.T

    cx, cy, w, h, conf = pred[:, 0], pred[:, 1], pred[:, 2], pred[:, 3], pred[:, 4]
    mask = conf > conf_thresh
    if not mask.any():
        return None

    best_idx = np.where(mask)[0][np.argmax(conf[mask])]
    x1 = cx[best_idx] - w[best_idx] / 2
    y1 = cy[best_idx] - h[best_idx] / 2
    x2 = cx[best_idx] + w[best_idx] / 2
    y2 = cy[best_idx] + h[best_idx] / 2
    return [x1, y1, x2, y2]


# ============================================================
# Crop + Resize trên GPU (giống resize_with_padding nhưng trên GPU)
# ============================================================
def crop_and_prepare_gpu(img_gpu, box_640):
    """
    img_gpu: (3, 640, 640) trên GPU
    Returns: (1, 3, 40, 40) contiguous GPU tensor, normalized
    """
    if box_640 is not None:
        x1 = max(0, int(box_640[0]))
        y1 = max(0, int(box_640[1]))
        x2 = min(640, int(box_640[2]))
        y2 = min(640, int(box_640[3]))
        if x2 > x1 and y2 > y1:
            crop = img_gpu[:, y1:y2, x1:x2]
        else:
            crop = img_gpu
    else:
        crop = img_gpu

    # ★ Resize giữ aspect ratio + padding (giống resize_with_padding)
    C, H, W = crop.shape
    scale = min(40 / W, 40 / H)
    new_w, new_h = int(W * scale), int(H * scale)

    resized = F.interpolate(
        crop.unsqueeze(0), size=(new_h, new_w),
        mode='bilinear', align_corners=False
    )

    # Center padding
    pad_w = 40 - new_w
    pad_h = 40 - new_h
    padded = F.pad(
        resized,
        (pad_w // 2, pad_w - pad_w // 2, pad_h // 2, pad_h - pad_h // 2),
        mode='constant', value=0
    )

    # Normalize
    normalized = (padded - d_mean) / d_std
    return normalized.contiguous()  # (1, 3, 40, 40)


# ============================================================
# Full 2-stage TRT inference (tối ưu)
# ============================================================
def trt_infer_2stage(img_tensor):
    np.copyto(h_yolo_in, img_tensor.numpy())
    cuda.memcpy_htod_async(d_yolo_in, h_yolo_in, stream)

    yolo_context.execute_async_v3(stream.handle)
    cuda.memcpy_dtoh_async(h_yolo_out, d_yolo_out, stream)
    stream.synchronize()

    box = postprocess_yolo(h_yolo_out)

    # ★ SỬA: reuse pinned memory, không gọi img_tensor.cuda() thừa
    img_gpu = torch.from_numpy(h_yolo_in).cuda()
    rvit_input = crop_and_prepare_gpu(img_gpu[0], box)

    cuda.memcpy_dtod_async(d_rvit_in, rvit_input.data_ptr(), 1*3*40*40*4, stream)
    rvit_context.execute_async_v3(stream.handle)
    cuda.memcpy_dtoh_async(h_rvit_out, d_rvit_out, stream)
    stream.synchronize()

    return h_rvit_out[0].argmax(-1).tolist()


# ============================================================
# Evaluate (giống end-to-end)
# ============================================================
test_loader_b1 = DataLoader(
    val_dataset, batch_size=1, shuffle=False,
    num_workers=2, collate_fn=LicensePlateDataset.collate_fn
)

correct_seq, total_seq = 0, 0
correct_chars, total_chars = 0, 0

for i, (img, target) in enumerate(test_loader_b1):
    pred_ids = trt_infer_2stage(img)

    true_ids = target[0, 1:].tolist()
    pred_content, true_content = extract_pred_and_true(pred_ids, true_ids)

    correct, total = compute_crr(pred_content, true_content)
    correct_chars += correct
    total_chars += total

    if pred_content == true_content:
        correct_seq += 1
    total_seq += 1

    if i < 10:
        status = "✓" if pred_content == true_content else "✗"
        print(f"  {status} GT='{index_to_char(true_ids)}' TRT='{index_to_char(pred_ids)}'")

print(f"\n{'='*50}")
print(f"TensorRT 2-Stage Results:")
print(f"  CRR:          {correct_chars/total_chars:.4f}")
print(f"  E2E Accuracy: {correct_seq/total_seq:.4f} ({correct_seq}/{total_seq})")
print(f"{'='*50}")


# ============================================================
# Benchmark FPS (giống end-to-end)
# ============================================================
benchmark_loader = DataLoader(
    val_dataset, batch_size=1, shuffle=False,
    num_workers=2, collate_fn=LicensePlateDataset.collate_fn
)

for i, (imgs, _) in enumerate(benchmark_loader):
    trt_infer_2stage(imgs)
    if i >= 49:
        break

latencies_trt = []
for imgs, _ in benchmark_loader:
    start_event = cuda.Event()
    end_event = cuda.Event()

    start_event.record(stream)
    trt_infer_2stage(imgs)
    end_event.record(stream)
    end_event.synchronize()

    latencies_trt.append(start_event.time_till(end_event))

latencies_trt = np.array(latencies_trt)

print(f"\n{'='*50}")
print(f"TensorRT 2-Stage Speed (batch=1, N={len(latencies_trt)}):")
print(f"  GPU: {cuda.Device(0).name()}")
print(f"  Mean ± Std: {latencies_trt.mean():.2f} ± {latencies_trt.std():.2f} ms")
print(f"  Median:     {np.median(latencies_trt):.2f} ms")
print(f"  FPS:        {1000/latencies_trt.mean():.1f}")
print(f"{'='*50}")

YOLO: images → output0 (1, 5, 8400)
RViT: crop → ocr_logits (1, 13, 39)
  ✓ GT='61P3452' TRT='61P3452'
  ✓ GT='51R15464' TRT='51R15464'
  ✗ GT='60C37034' TRT='51C37034'
  ✗ GT='60C15754' TRT='60C12734'
  ✓ GT='61C13025' TRT='61C13025'
  ✗ GT='51R17320' TRT='61R17320'
  ✓ GT='49R00228' TRT='49R00228'
  ✗ GT='72C07256' TRT='72C07755'
  ✓ GT='49R00200' TRT='49R00200'
  ✓ GT='49C09817' TRT='49C09817'

TensorRT 2-Stage Results:
  CRR:          0.8332
  E2E Accuracy: 0.4742 (836/1763)

TensorRT 2-Stage Speed (batch=1, N=1763):
  GPU: Tesla T4
  Mean ± Std: 10.32 ± 0.38 ms
  Median:     10.26 ms
  FPS:        96.9


In [7]:
# ============================================================================
# 📊 PRINT TABLE METRICS
# ============================================================================

print("\n" + "="*90)
print(f"{'TABLE 1: YOLO-RViT Metrics (640x640, batch=1, T4 GPU)':^90}")
print("="*90)

# Header
print(f"{'Model':<15} {'Params(M)':>10} {'GFLOPs':>10} {'Size(MB) FP32':>14} {'Size(MB) FP16':>14} {'Lat_FP32(ms)':>14} {'FPS_FP32':>10} {'Lat_FP16(ms)':>14} {'FPS_FP16':>10} {'Lat_TRT(ms)':>13} {'FPS_TRT':>10}")
print("-"*90)

# Data row 
print(f"{'YOLO-RViT':<15} "
      f"{total_params/1e6:>10.2f} "
      f"{total_flops/1e9:>10.2f} "
      f"{model_size_fp32_mb:>14.2f} "
      f"None "
      f"{mean_latency_fp32:>14.2f} "
      f"{fps_fp32:>10.1f} "
      f"None "
      f"None "
      f"{latencies_trt.mean():>13.2f} "
      f"{1000/latencies_trt.mean():>10.1f}")

print("="*90)


                  TABLE 1: YOLO-RViT Metrics (640x640, batch=1, T4 GPU)                   
Model            Params(M)     GFLOPs  Size(MB) FP32  Size(MB) FP16   Lat_FP32(ms)   FPS_FP32   Lat_FP16(ms)   FPS_FP16   Lat_TRT(ms)    FPS_TRT
------------------------------------------------------------------------------------------
YOLO-RViT            23.97      36.66          91.43 None          40.33       24.8 None None         10.32       96.9
